In [66]:
import cvxpy as cp
import numpy as np
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
import random
import math

In [67]:
import random
import math
import numpy as np
import cvxpy as cp

# Generate nodes and edges
def make_network(n):
    nodes = [i for i in range(1, n + 1)]
    edges = [(i, i + 1) for i in range(1, n)] + [(n, 1)]
    return nodes, edges

# Generate random reactances for edges
def gen_react(edge_list, low, high):
    edge_reactances = {}
    for edge in edge_list:
        edge_reactances[edge] = random.uniform(low, high)
    return edge_reactances

# Generate random phases for nodes
def gen_phases(node_list):
    return [random.uniform(-math.pi / 18, math.pi / 18) for _ in node_list]

def compute_power(edge_list, phases, reactances):
    flows = []
    for i, j in edge_list:
        flows.append((phases[i - 1] - phases[j - 1]) / reactances[(i, j)])
    return flows

def compute_nodal_power(node_list, power_flows):
    injections = []
    injections.append(power_flows[0] + power_flows[-1])
    for i in range(1, len(node_list) - 1):
        injections.append(power_flows[i - 1] + power_flows[i])
    injections.append(power_flows[-2] + power_flows[-1])
    return injections

# add  sensors to nodes and simulate attacks
def node_sensors(node_list, true_powers):
    attacked_nodes = [i for i in range(5, len(node_list), 10)]
    noisy_measurements = []
    for i in node_list:
        noise = np.random.normal(0, 0.001)
        attack = np.random.uniform(-10, 10) if i in attacked_nodes else 0
        noisy_measurements.append(true_powers[i - 1] + noise + attack)
    return noisy_measurements

# add sensors to edges and simulate attacks
def edge_sensors(edge_list, true_flows):
    attacked_edges = [(i, i + 1) for i in range(10, 100, 10)] + [(100, 1)]
    noisy_measurements = []
    for idx, (i, j) in enumerate(edge_list):
        noise = np.random.normal(0, 0.001)
        attack = np.random.uniform(-10, 10) if (i, j) in attacked_edges else 0
        noisy_measurements.append(true_flows[idx] + noise + attack)
    return noisy_measurements

# Check success of attack detection 
def validate_detection(node_attacks, edge_attacks, estimated_vi, estimated_vij, nodes, edges):
    for i in nodes:
        if (i in node_attacks and abs(estimated_vi[i - 1]) <= 0.01) or \
           (i not in node_attacks and abs(estimated_vi[i - 1]) > 0.01):
            return False
    for idx, edge in enumerate(edges):
        if (edge in edge_attacks and abs(estimated_vij[idx]) <= 0.01) or \
           (edge not in edge_attacks and abs(estimated_vij[idx]) > 0.01):
            return False
    return True

successful_detections = 0
num_trials = 100

for _ in range(num_trials):
    nodes, edges = make_network(100)
    reactances = gen_react(edges, 0.1, 0.2)
    phases = gen_phases(nodes)
    power_flows = compute_power(edges, phases, reactances)
    power_injections = compute_nodal_power(nodes, power_flows)

    node_measurements = node_sensors(nodes, power_injections)
    edge_measurements = edge_sensors(edges, power_flows)

    # solve problem
    phase_estimates = cp.Variable(len(nodes))
    noise_node = cp.Variable(len(nodes))
    noise_edge = cp.Variable(len(edges))
    attack_node = cp.Variable(len(nodes))
    attack_edge = cp.Variable(len(edges))

    constraints = []

    # define nodal and edge constraints
    estimated_flows = compute_power(edges, phase_estimates, reactances)
    estimated_injections = compute_nodal_power(nodes, estimated_flows)

    for i in range(len(nodes)):
        constraints.append(node_measurements[i] == estimated_injections[i] + noise_node[i] + attack_node[i])

    for i in range(len(edges)):
        constraints.append(edge_measurements[i] == estimated_flows[i] + noise_edge[i] + attack_edge[i])

    constraints.append(phase_estimates[0] == 0)

    objective = cp.Minimize(
        cp.norm(noise_node, 2)**2 +
        cp.norm(noise_edge, 2)**2 +
        0.01 * cp.norm(attack_node, 1) +
        0.01 * cp.norm(attack_edge, 1)
    )

    problem = cp.Problem(objective, constraints)
    problem.solve()

    # check attack detection
    detected_node_attacks = attack_node.value
    detected_edge_attacks = attack_edge.value
    node_attacks = [i for i in range(5, 100, 10)]
    edge_attacks = [(i, i + 1) for i in range(10, 100, 10)] + [(100, 1)]

    if validate_detection(node_attacks, edge_attacks, detected_node_attacks, detected_edge_attacks, nodes, edges):
        successful_detections += 1

print(f"Success rate: {successful_detections / num_trials}")


Success rate: 0.98
